# Lasso Regression

## 1. Objective

Evaluate Lasso using the same training data, preprocessing, and five cross-validation folds used for Linear Regression and Ridge. Select alpha using mean validation RMSE, compare all three models, and inspect how many coefficients Lasso sets to zero.

The held-out test set remains reserved for final evaluation after model selection.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import joblib

from sklearn.base import clone
from sklearn.linear_model import Lasso
from sklearn.model_selection import KFold, GridSearchCV
from sklearn.pipeline import Pipeline

In [2]:
project_root = Path.cwd().parent

X_train = pd.read_csv(
    project_root / "data/processed/X_train.csv",
    index_col="row_index",
    dtype={"Postcode": str}
)

y_train = pd.read_csv(
    project_root / "data/processed/y_train.csv",
    index_col="row_index"
)["Price"]

preprocessor = joblib.load(
    project_root / "models/preprocessor_unfitted.joblib"
)

assert X_train.index.equals(y_train.index)
assert X_train.index.is_unique
assert y_train.notna().all()

print("Predictors:", X_train.shape)
print("Target:", y_train.shape)

Predictors: (10864, 16)
Target: (10864,)


## 2. Tune Lasso with Cross-Validation

Fit preprocessing within each training fold and evaluate six alpha values using the same five folds as the earlier models. Select alpha by mean validation RMSE and retain MAE and R² for comparison.

In [4]:
cv = KFold(n_splits=5, shuffle=True, random_state=42)
cv_splits = list(cv.split(X_train, y_train))

scoring = {
    "MAE": "neg_mean_absolute_error",
    "RMSE": "neg_root_mean_squared_error",
    "R2": "r2"
}

lasso_pipeline = Pipeline([
    ("preprocessor", clone(preprocessor)),
    ("model", Lasso(
        max_iter=200_000,
        tol=1e-4,
        selection="random",
        random_state=42
    ))
])

lasso_search = GridSearchCV(
    estimator=lasso_pipeline,
    param_grid={
        "model__alpha": [1, 10, 100, 1000, 10000, 100000]
    },
    scoring=scoring,
    cv=cv_splits,
    refit="RMSE",
    return_train_score=True,
    error_score="raise",
    n_jobs=2,
    verbose=2
)

lasso_search.fit(X_train, y_train)

print("Best alpha:", lasso_search.best_params_["model__alpha"])
print(
    "Mean validation RMSE:",
    round(-lasso_search.best_score_, 2)
)

Fitting 5 folds for each of 6 candidates, totalling 30 fits
Best alpha: 100
Mean validation RMSE: 390142.86


## 3. Compare Linear Regression, Ridge, and Lasso

Compare mean validation MAE, RMSE, and R² using the same five cross-validation folds. These development results guide model selection; the held-out test set remains unused.

In [5]:
tables_dir = project_root / "results/tables"

lasso_results = pd.DataFrame(lasso_search.cv_results_)
best = lasso_results.iloc[lasso_search.best_index_]

lasso_mean_scores = pd.Series({
    "MAE": -best["mean_test_MAE"],
    "RMSE": -best["mean_test_RMSE"],
    "R2": best["mean_test_R2"]
})

linear_summary = pd.read_csv(
    tables_dir / "linear_cv_summary.csv",
    index_col="Metric"
)

ridge_summary = pd.read_csv(
    tables_dir / "ridge_cv_summary.csv",
    index_col="Metric"
)

model_comparison = pd.DataFrame({
    "Linear Regression": linear_summary["mean"],
    "Ridge": ridge_summary["mean"],
    "Lasso": lasso_mean_scores
}).T

model_comparison.round(3)

,MAE,RMSE,R2
Linear Regression,242395.379,395384.443,0.618
Ridge,237848.212,388385.431,0.631
Lasso,238895.731,390142.860,0.627


## 4. Inspect Coefficient Sparsity

Count zero and non-zero coefficients in the selected Lasso pipeline fitted on the full training dataset. These coefficients correspond to transformed numerical features and one-hot encoded categories.

In [6]:
best_lasso_pipeline = lasso_search.best_estimator_

lasso_coefficients = pd.DataFrame({
    "Feature": best_lasso_pipeline.named_steps[
        "preprocessor"
    ].get_feature_names_out(),
    "Coefficient": best_lasso_pipeline.named_steps["model"].coef_
})

zero_count = (lasso_coefficients["Coefficient"] == 0).sum()
total_count = len(lasso_coefficients)

print("Total coefficients:", total_count)
print("Zero coefficients:", zero_count)
print("Non-zero coefficients:", total_count - zero_count)
print("Zero percentage:", round(zero_count / total_count * 100, 2))

Total coefficients: 802
Zero coefficients: 481
Non-zero coefficients: 321
Zero percentage: 59.98


### Observation

At alpha = 100, Lasso set 481 of 802 transformed-feature coefficients to exactly zero (59.98%), leaving 321 non-zero coefficients. This produces a sparser model, although Ridge achieved a lower mean validation RMSE. Zero coefficients refer to individual transformed columns, including one-hot encoded categories, and do not establish that the corresponding original variables are universally unimportant.

## 5. Save Results and the Fitted Pipeline

Save the cross-validation search results, three-model comparison, coefficients, and fitted Lasso pipeline. The pipeline expects the same prepared predictor columns used during training.

In [8]:
models_dir = project_root / "models"
models_dir.mkdir(parents=True, exist_ok=True)
tables_dir.mkdir(parents=True, exist_ok=True)

lasso_results.to_csv(
    tables_dir / "lasso_search_results.csv",
    index=False
)

model_comparison.to_csv(
    tables_dir / "linear_ridge_lasso_cv_comparison.csv",
    index_label="Model"
)

lasso_coefficients.to_csv(
    tables_dir / "lasso_coefficients.csv",
    index=False
)

joblib.dump(
    best_lasso_pipeline,
    models_dir / "lasso_pipeline_fitted.joblib"
)

print("Lasso results and fitted pipeline saved.")

Lasso results and fitted pipeline saved.


In [9]:
loaded_lasso = joblib.load(
    models_dir / "lasso_pipeline_fitted.joblib"
)

sample = X_train.head()

print(
    "Predictions match:",
    np.allclose(
        best_lasso_pipeline.predict(sample),
        loaded_lasso.predict(sample)
    )
)

Predictions match: True


## 6. Conclusion

Lasso selected alpha = 100 from the tested grid, achieving mean validation MAE of 238,896, RMSE of 390,143, and R² of 0.627. It set 481 of 802 transformed-feature coefficients to zero, producing a sparser model.

Ridge achieved the lowest mean validation RMSE among Linear Regression, Ridge, and Lasso and is selected for final held-out evaluation. The comparison uses development cross-validation results, including the folds used for hyperparameter selection.

The fitted Lasso pipeline and supporting results were saved. The held-out test set has not been used for model selection.